In [ ]:
# ===========================================================================
# RESULTS - interactive driver (publication export)
#
# Aggregates every experiment notebooks/03_training.ipynb has produced,
# tests whether the differences between models are real (not just LOSO fold
# noise), and exports paper-ready tables and figures. Nothing here
# recomputes a metric - it only aggregates, tests, and reformats what
# src/training/reporting.py already wrote per run.
#
#   src/results.py   aggregation, paired significance testing, ROC/PR
#                     overlay, publication styling, paper export
#
# PREREQUISITES: at least one trained run from notebooks/03_training.ipynb
# (outputs/metrics/*.summary.csv, *.per_fold.csv).
# ===========================================================================

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config
from src.console import print_header, print_kv
from src.results import set_publication_style

config.ensure_directories()
set_publication_style()

print_header("Results Notebook")
print_kv("Metrics directory", config.METRICS_DIR)

In [ ]:
# STAGE 1 - Aggregate all experiment outputs. Every trained run's pooled mean
# metrics in one table, read straight off the *.summary.csv files
# aggregate_fold_metrics() already wrote per run - the single place to see
# every experiment run so far, detection and severity together.
#
# style_comparison_table() colour-grades each metric column (red->green,
# best value per column bolded) so the winner is visible at a glance instead
# of requiring a scan across decimals - test_loss is coloured the opposite
# way (green = lowest) since it's the one column here where lower is better.
from src.results import load_all_experiment_summaries, style_comparison_table

all_summaries = load_all_experiment_summaries()
print_header("All Experiment Summaries")

loss_cols = [c for c in all_summaries.columns if "loss" in c]
score_cols = [c for c in all_summaries.columns if c not in loss_cols]
display(style_comparison_table(all_summaries[score_cols]))
if loss_cols:
    display(style_comparison_table(all_summaries[loss_cols], higher_is_better=False))

In [ ]:
# STAGE 2 - Statistical significance tests. Is a fusion refinement actually
# better than the model it claims to improve on, or within LOSO fold noise?
# Paired Wilcoxon signed-rank test, fold-matched, on F1 - the same metric
# notebooks/03_training.ipynb's severity gating ranks by. Compares the two
# Phase 6 attention-fusion variants (Models E, F) against the plain
# concatenated 'fusion' run (Model D), since that is the pre-attention
# baseline each of them claims to improve on.
from src.results import compare_models_statistically

BASELINE_RUN = "detection_fusion"
CANDIDATE_RUNS = ["detection_attention_fusion", "detection_attention_fusion_praat"]

significance_results = []
for candidate in CANDIDATE_RUNS:
    try:
        significance_results.append(
            compare_models_statistically(BASELINE_RUN, candidate, metric="f1"))
    except (FileNotFoundError, ValueError) as e:
        print_kv("Skipped", f"{candidate}: {e}")

significance_df = pd.DataFrame(significance_results)
if not significance_df.empty:
    significance_path = config.METRICS_DIR / "significance_tests.csv"
    significance_df.to_csv(significance_path, index=False)
    print_kv("Saved", significance_path)
significance_df

In [ ]:
# STAGE 3 - Detection comparison table, formatted for the paper. Re-presents
# the table notebooks/03_training.ipynb already built and saved
# (phase2_comparison.csv) - this notebook doesn't regenerate it, only
# formats it for publication. Colour-graded the same way as Stage 1.
from src.results import style_comparison_table

detection_table = pd.read_csv(config.METRICS_DIR / "phase2_comparison.csv", index_col=0)
print_header("Detection Comparison")

loss_cols = [c for c in detection_table.columns if "loss" in c]
score_cols = [c for c in detection_table.columns if c not in loss_cols]
display(style_comparison_table(detection_table[score_cols]))
if loss_cols:
    display(style_comparison_table(detection_table[loss_cols], higher_is_better=False))

In [ ]:
# STAGE 4 - Severity comparison table, formatted the same way, if
# notebooks/03_training.ipynb's severity stages have been run.
severity_path = config.METRICS_DIR / "phase3_severity_comparison.csv"
if severity_path.exists():
    severity_table = pd.read_csv(severity_path, index_col=0)
    print_header("Severity Comparison")
    loss_cols = [c for c in severity_table.columns if "loss" in c]
    score_cols = [c for c in severity_table.columns if c not in loss_cols]
    display(style_comparison_table(severity_table[score_cols]))
    if loss_cols:
        display(style_comparison_table(severity_table[loss_cols], higher_is_better=False))
else:
    severity_table = None
    print_kv("Severity comparison",
             f"not found at {severity_path} - run notebooks/03_training.ipynb Stages 10-13")

In [ ]:
# STAGE 5 - ROC and PR curves, overlaid across every trained detection
# variant on one figure each - the publication-facing, multi-model
# counterpart to the per-fold ROC curve each run already saves
# (outputs/roc/<run>/ALL_FOLDS_pooled.png).
from src.results import plot_roc_pr_comparison

DETECTION_RUNS = [name for name in detection_table.index if name != "baseline_svm"]

curve_paths = plot_roc_pr_comparison(DETECTION_RUNS, task="detection", show=True)
print_header("ROC / PR Comparison")
print_kv("ROC figure", curve_paths["roc"])
print_kv("PR figure", curve_paths["pr"])

In [ ]:
# STAGE 6 - Publication-quality figures. Stage 0's set_publication_style()
# is already active for every figure in this notebook; this stage produces
# the two "hero" figures a paper draft actually needs - the six-variant
# ablation bar chart and the best model's pooled confusion matrix - redrawn
# here (not just linked) so they pick up the serif/IEEE-draft styling.
import matplotlib.pyplot as plt

from src.model_analysis import plot_ablation_comparison

BEST_RUN = detection_table.drop(index="baseline_svm", errors="ignore")["f1"].idxmax()

ablation_figure = plot_ablation_comparison(
    detection_table, title="Ablation Comparison - Detection", show=True)

cm_path = config.CONFUSION_MATRIX_DIR / BEST_RUN / "ALL_FOLDS_pooled.png"
if cm_path.exists():
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(plt.imread(cm_path))
    ax.axis("off")
    ax.set_title(f"{BEST_RUN} — pooled confusion matrix")
    plt.show()

print_header("Publication Figures")
print_kv("Ablation figure", ablation_figure)
print_kv("Best run (by F1)", BEST_RUN)

In [ ]:
# STAGE 7 - Export all results for the paper. Copies the key tables/figures
# into outputs/paper_exports/ and writes LaTeX table snippets alongside them
# - the single folder to hand off when writing the paper.
from src.results import export_results_for_paper

export_dir = export_results_for_paper(run_names=DETECTION_RUNS)
print_header("Paper Export")
print_kv("Export directory", export_dir)